<a href="https://colab.research.google.com/github/asomers205/ds2002fa26/blob/main/notebooks/05-etl-walmart/2026-10-14%20%E2%80%94%20Mini-ETL%20on%20Walmart%20Sample%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.


*(One correction to the line above, taken from the actual extract report: only `unit_price` arrived as text. `quantity` arrived as `float64`, not `int64`, because pandas reads the literal `NULL` as missing by default. The report shows two nulls, not one: the blank timestamp and that `NULL` quantity.)*

**Problems visible in these twelve rows** (thirteen found; at least six were promised):

1. **Exact duplicate row.** `TXN1` appears twice, identical in every column: a double scan.
2. **`store_id` in five spellings for two stores.** `FL-105` / `fl-105`, and `FL-239` / `fl-239` / `FL 239`. The space variant means upper-casing alone does not fix it.
3. **`timestamp` in three formats.** ISO with a space (`2024-09-06 10:06:00`), ISO with a `T` (`2024-09-06T11:00:00`), and US `MM/DD/YYYY HH:MM` with no seconds (`09/06/2024 10:40`).
4. **Missing `timestamp`.** `TXN8` is blank; it becomes `NaT` and cannot take part in any by-date view.
5. **Fragmented SKUs for one product.** `POPTART-STRAW`, `PT-12` and `SKU#459812` are all strawberry Pop-Tarts. One of them contains a `#`.
6. **Fragmented SKUs for bottled water.** `WATER-24` and `BTL-WATER-24`.
7. **A look-alike that is a different product.** `POPTART-BLUE` (blueberry) must not be merged with strawberry.
8. **`product_name` inconsistent in case, plural and word order.** `Pop-Tarts Strawberry` / `pop-tarts strawberry` / `Strawberry Pop-Tart`; `Bottled Water` / `bottled water` / `Bottled Water 24pk`.
9. **`category` inconsistent in case.** `Breakfast` / `breakfast`, `Grocery` / `grocery`.
10. **Missing `quantity`.** `TXN9` (flashlight) has the literal `NULL`.
11. **Negative `quantity`.** `TXN3` has `-1` bottled water: a return or void, not a purchase.
12. **`unit_price` mixed formatting.** Some values carry a `$` (`$4.11`), some do not (`4.11`), so the column is text.
13. **Two prices for one SKU.** `PT-12` sells at `3.86` in `TXN2` and `$4.11` in `TXN8`, so after consolidation one product carries two prices. Not necessarily an error, but it has to be known before revenue is trusted.

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [3]:
def _normalize_store_id(s):
    """'fl-105', 'FL 239', ' fl_239 ' -> 'FL-105', 'FL-239', 'FL-239'."""
    return (s.astype(str).str.strip().str.upper()
             .str.replace(r'[\s_]+', '-', regex=True))


def transform_types(df):
    """Dedupe and coerce types. Returns a NEW frame; `df` is never modified.

    Decisions recorded here (reasoning in the markdown cell below):
      - exact duplicate rows dropped, and dropped FIRST, before anything is renamed
      - unit_price: '$' stripped, cast to float
      - quantity: cast to numeric; rows with a MISSING quantity are dropped and
        reported; rows with quantity <= 0 (returns / voids) are dropped and reported
      - store_id: trimmed, upper-cased, space or underscore -> hyphen, then checked
      - timestamp: parsed with format='mixed' (three formats in the file);
        blank -> NaT, row is KEPT and reported
      - category: trimmed and title-cased; every other text column trimmed
    """
    out = df.copy()
    n0 = len(out)

    # 1. dedupe FIRST: an exact duplicate is only detectable before anything is renamed
    out = out.drop_duplicates()
    n_dupes = n0 - len(out)
    assert out['transaction_id'].is_unique, \
        'a transaction_id repeats with different contents: investigate, do not drop'

    # 2. whitespace and case hygiene on the text columns
    for col in out.select_dtypes(include='object').columns:
        out[col] = out[col].str.strip()
    out['category'] = out['category'].str.title()

    # 3. store_id -> one consistent format, then prove it
    stores_before = sorted(out['store_id'].unique())
    out['store_id'] = _normalize_store_id(out['store_id'])
    assert out['store_id'].str.fullmatch(r'[A-Z]{2}-\d+').all(), \
        f'unexpected store_id spelling: {sorted(out["store_id"].unique())}'

    # 4. unit_price -> float ('$4.11' and '4.11' both appear)
    out['unit_price'] = pd.to_numeric(
        out['unit_price'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    assert out['unit_price'].notna().all(), 'unparseable unit_price'

    # 5. quantity -> numeric, then the decision on null and negative
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    is_null = out['quantity'].isna()
    is_nonpos = out['quantity'] <= 0
    rejected = out[is_null | is_nonpos].assign(
        reject_reason=lambda d: d['quantity'].isna().map(
            {True: 'quantity missing', False: 'quantity <= 0 (return / void)'}))
    out = out[~(is_null | is_nonpos)].copy()
    out['quantity'] = out['quantity'].astype('int64')

    # 6. timestamp -> datetime; blank -> NaT (row kept, reported)
    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce', format='mixed')
    no_ts = out.loc[out['timestamp'].isna(), 'transaction_id'].tolist()

    out = out.reset_index(drop=True)

    print(f'transform_types: {n0} -> {len(out)} rows')
    print(f'  dropped {n_dupes} exact duplicate row(s)')
    for _, r in rejected.iterrows():
        print(f'  dropped {r.transaction_id}: {r.reject_reason} '
              f'(sku={r.sku}, quantity={r.quantity})')
    print(f'  store_id {stores_before} -> {sorted(out["store_id"].unique())}')
    print(f'  timestamp blank, kept as NaT: {no_ts}')
    return out


typed = transform_types(raw)
pd.testing.assert_frame_equal(raw, pd.read_csv(StringIO(RAW)))   # the input was not touched
print('rows:', len(raw), '->', len(typed))
typed.dtypes

transform_types: 12 -> 9 rows
  dropped 1 exact duplicate row(s)
  dropped TXN3: quantity <= 0 (return / void) (sku=WATER-24, quantity=-1.0)
  dropped TXN9: quantity missing (sku=FLASH-AA, quantity=nan)
  store_id ['FL 239', 'FL-105', 'FL-239', 'fl-105', 'fl-239'] -> ['FL-105', 'FL-239']
  timestamp blank, kept as NaT: ['TXN8']
rows: 12 -> 9


transaction_id            object
store_id                  object
timestamp         datetime64[ns]
sku                       object
product_name              object
category                  object
quantity                   int64
unit_price               float64
dtype: object

**Decisions recorded in `transform_types`, and why:**

- **Dedupe before anything else.** `TXN1` is a true double scan: every column identical. If a `transaction_id` ever repeated with *different* contents that is not a double scan, so the function asserts instead of silently keeping one.
- **Null quantity (`TXN9`, flashlight): dropped and logged.** A sale with an unknown unit count cannot be counted. Imputing `1` would invent data, and leaving `NaN` in the clean table would let the row vanish from every `sum()` without anyone noticing, which is the quiet failure Monday's lecture warned about. Side effect worth knowing: this was the only flashlight row, so `FLASH-AA` is absent from the clean table. On the midterm, measure the share of null quantities first; if it is material, that is a source problem to raise, not rows to drop.
- **Negative quantity (`TXN3`, -1 water): dropped and logged as a return / void.** The pipeline measures what people *buy*. A return is not a purchase, and netting it against that day's sales would understate demand (the item may have been bought on a different day). It is treated as a genuine return, not as a sign error to flip to `+1`. On the midterm, returns deserve their own table rather than the bin.
- **Store ids:** trimmed, upper-cased, space or underscore replaced by a hyphen, then asserted against the `XX-nnn` pattern so a sixth spelling fails loudly instead of becoming a third store. There were five distinct strings for two stores, not four.
- **Timestamps:** `format='mixed'` parses the three formats row by row. The `MM/DD/YYYY` rows are read month-first (US convention), which agrees with the ISO rows in the same file (all September). `TXN8` has no timestamp: the row is **kept**, because its two units are real, but it cannot appear in any by-date view, and Build 6 says so explicitly.
- **Category** is title-cased so `breakfast` / `Breakfast` collapse. `product_name` is left exactly as it arrived; product identity is handled by `canonical_sku` in the next step, not by editing names.
- **`unit_price`:** `$` stripped, cast to float. The two prices on `PT-12` (3.86 and 4.11) are kept as they are. Price is a fact about the transaction, not about the product.

### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [4]:
CANON = {
    # strawberry Pop-Tarts: three codes, one product (same flavor, same $4.11 shelf price)
    'PT-12':         'POPTART-STRAW',
    'SKU#459812':    'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    # bottled water: 'Bottled Water' and 'Bottled Water 24pk' at the same $5.00;
    # the WATER-24 code itself says 24-pack, so these are one product
    'BTL-WATER-24':  'WATER-24',
    'WATER-24':      'WATER-24',
    # deliberately NOT mapped: POPTART-BLUE (different flavor is a different product), FLASH-AA
}


def consolidate(df, mapping=CANON):
    """Add canonical_sku. A SKU absent from the mapping keeps its own value, never NaN."""
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(mapping).fillna(out['sku'])
    # consolidation renames; it never adds, removes, or loses anything
    assert len(out) == len(df), 'consolidation changed the row count'
    assert out['canonical_sku'].notna().all(), 'an unmapped SKU became NaN'
    assert out['quantity'].sum() == df['quantity'].sum(), 'consolidation changed total units'
    print(f'consolidate: {df["sku"].nunique()} raw SKUs -> {out["canonical_sku"].nunique()} canonical')
    return out


consolidated = consolidate(typed)
assert consolidated['quantity'].sum() == typed['quantity'].sum()
print('total units unchanged:', typed['quantity'].sum(), '->', consolidated['quantity'].sum())
print()
print('units per canonical SKU:')
print(consolidated.groupby('canonical_sku')['quantity'].sum().sort_values(ascending=False).to_string())
print()
print('what was merged (so somebody can disagree with it):')
print(consolidated.groupby('canonical_sku')
      .agg(raw_skus=('sku', lambda s: ', '.join(sorted(set(s)))),
           raw_names=('product_name', lambda s: ', '.join(sorted(set(s)))))
      .to_string())

consolidate: 6 raw SKUs -> 3 canonical
total units unchanged: 28 -> 28

units per canonical SKU:
canonical_sku
POPTART-STRAW    17
WATER-24         10
POPTART-BLUE      1

what was merged (so somebody can disagree with it):
                                       raw_skus                                                        raw_names
canonical_sku                                                                                                   
POPTART-BLUE                       POPTART-BLUE                                              Pop-Tarts Blueberry
POPTART-STRAW  POPTART-STRAW, PT-12, SKU#459812  Pop-Tarts Strawberry, Strawberry Pop-Tart, pop-tarts strawberry
WATER-24                 BTL-WATER-24, WATER-24                                Bottled Water 24pk, bottled water


**Why blueberry stays separate:** flavor is the thing the case is about. The Walmart finding was not "Pop-Tarts sell before a hurricane", it was *strawberry* Pop-Tarts at seven times their normal rate. Merging flavors would erase exactly the signal the pipeline exists to find, and it would leave the clean table unable to answer any flavor-level question later. A different flavor is a different product (its own SKU in the real catalog, its own shelf facing, its own reorder) in a way that `PT-12` versus `POPTART-STRAW` is not: those are the *same* product under different codes. If the business later wants a brand-level view ("all Pop-Tarts"), that is an extra `product_family` column on top of `canonical_sku`, not a replacement for it.

The call least certain here is the water. `Bottled Water` with no pack size was merged with `Bottled Water 24pk` because the `WATER-24` code and the identical $5.00 price both say 24-pack. If a different pack size ever appears under that code, the mapping table is where to overrule this.

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [5]:
def derive(df):
    """Add revenue and a date column. Returns a new frame."""
    out = df.copy()
    out['revenue'] = (out['quantity'] * out['unit_price']).round(2)
    # .dt.normalize() keeps `date` a real datetime with the time zeroed out, so it
    # merges directly with a weather table keyed by date and round-trips through SQLite.
    out['date'] = out['timestamp'].dt.normalize()
    assert out['revenue'].notna().all(), 'revenue could not be computed for every row'
    print(f'derive: revenue total {out["revenue"].sum():.2f}; '
          f'{int(out["date"].isna().sum())} row(s) have no date')
    return out


enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

derive: revenue total 123.73; 1 row(s) have no date


,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2,8.22,2024-09-06
1,TXN2,POPTART-STRAW,1,3.86,2024-09-06
2,TXN4,POPTART-STRAW,3,12.33,2024-09-06
3,TXN5,WATER-24,4,20.00,2024-09-06
4,TXN6,WATER-24,6,30.00,2024-09-07
5,TXN7,POPTART-STRAW,5,20.55,2024-09-07
6,TXN8,POPTART-STRAW,2,8.22,NaT
7,TXN10,POPTART-BLUE,1,4.11,2024-09-07
8,TXN11,POPTART-STRAW,4,16.44,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [6]:
def load(df, conn, table='clean_sales'):
    """Write df to `table` (replacing it), then read back and verify the load."""
    df.to_sql(table, conn, index=False, if_exists='replace')

    check = pd.read_sql_query(
        f'SELECT COUNT(*) AS n, SUM(quantity) AS units, SUM(revenue) AS revenue FROM {table}',
        conn)
    n_db = int(check['n'].iloc[0])
    units_db = int(check['units'].iloc[0])
    rev_db = float(check['revenue'].iloc[0])
    assert n_db == len(df), f'{table}: {n_db} rows in SQLite, expected {len(df)}'
    assert units_db == df['quantity'].sum(), f'{table}: units changed in the load'
    assert abs(rev_db - df['revenue'].sum()) < 0.005, f'{table}: revenue changed in the load'
    print(f'load: {n_db} rows written to {table}, read back {n_db} rows, '
          f'{units_db} units, revenue {rev_db:.2f}: verified')
    return n_db


conn = sqlite3.connect(':memory:')
load(enriched, conn)

load: 9 rows written to clean_sales, read back 9 rows, 28 units, revenue 123.73: verified


9

### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [7]:
import contextlib, io


def run_pipeline(conn, verbose=False):
    """extract -> transform_types -> consolidate -> derive -> load. Returns the loaded frame."""
    quiet = contextlib.nullcontext() if verbose else contextlib.redirect_stdout(io.StringIO())
    with quiet:
        raw = extract()
        enriched = derive(consolidate(transform_types(raw)))
        load(enriched, conn)
    return enriched


first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))

assert first.shape == second.shape, 'shape differs between runs'
assert first['revenue'].sum() == second['revenue'].sum(), 'revenue differs between runs'
pd.testing.assert_frame_equal(first, second)        # stronger: every cell identical
print(f'two fresh connections: shape {first.shape}, revenue {first["revenue"].sum():.2f}, both times')

# the other half of idempotency: re-running against the SAME connection must not grow the table
same_conn = sqlite3.connect(':memory:')
run_pipeline(same_conn)
run_pipeline(same_conn)
n = int(pd.read_sql_query('SELECT COUNT(*) AS n FROM clean_sales', same_conn)['n'].iloc[0])
assert n == len(first), f'clean_sales grew to {n} rows on re-run'
print(f'same connection twice: clean_sales still has {n} rows (if_exists="replace" held)')

two fresh connections: shape (9, 11), revenue 123.73, both times
same connection twice: clean_sales still has 9 rows (if_exists="replace" held)


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [8]:
STRAW = 'POPTART-STRAW'
straw = enriched.loc[enriched['canonical_sku'] == STRAW].copy()
straw['day'] = straw['date'].dt.strftime('%Y-%m-%d (%a)')     # NaT -> NaN, so it drops out of the groupby

by_date = straw.groupby('day')['quantity'].agg(units='sum', transactions='count')
print(f'{STRAW}, both stores, units by day:')
print(by_date.to_string())

print('\nby store:')
print(straw.pivot_table(index='day', columns='store_id', values='quantity',
                        aggfunc='sum', fill_value=0).to_string())

undated = straw[straw['day'].isna()]
print(f'\nnot in either table: {len(undated)} transaction(s) with no timestamp, '
      f'{undated["quantity"].sum()} of {straw["quantity"].sum()} strawberry units '
      f'({", ".join(undated["transaction_id"])})')

d1, d2 = by_date['units'].iloc[0], by_date['units'].iloc[-1]
print(f'\n{by_date.index[0]} -> {by_date.index[-1]}: {d1} -> {d2} units ({d2 / d1 - 1:+.0%})')
print('Rose? Yes, in this sample. Enough to conclude anything? No. See below.')

POPTART-STRAW, both stores, units by day:
                  units  transactions
day                                  
2024-09-06 (Fri)      6             3
2024-09-07 (Sat)      9             2

by store:
store_id          FL-105  FL-239
day                             
2024-09-06 (Fri)       3       3
2024-09-07 (Sat)       9       0

not in either table: 1 transaction(s) with no timestamp, 2 of 17 strawberry units (TXN8)

2024-09-06 (Fri) -> 2024-09-07 (Sat): 6 -> 9 units (+50%)
Rose? Yes, in this sample. Enough to conclude anything? No. See below.


**Verdict:** units went from 6 on Friday 2024-09-06 to 9 on Saturday 2024-09-07, with 2 more undated units that cannot be placed on either day. That is a rise, and it is not evidence of anything:

- **No baseline.** "Seven times the normal rate" needs a normal rate. Two days with no calm-weather weeks to compare against cannot separate a storm effect from ordinary day-to-day noise.
- **Friday versus Saturday.** Saturday is routinely the bigger grocery day. A day-of-week effect alone could produce this.
- **Five transactions.** Three on one day, two on the other. One basket of four boxes flips the conclusion either way.
- **All of the growth is one store.** `FL-105` goes 3 to 9; `FL-239` goes 3 to 0 on the days we can see. That looks more like where the dated rows happened to fall than like a demand signal.
- **One missing timestamp.** `TXN8` holds 2 of the 17 strawberry units and could belong to either day.

On the midterm the comparison that means something is units per store-day in the days before landfall, against the same stores' typical units per day in weeks with no storm.

---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [9]:
# Checkpoint
rows_after_transform = len(typed)                     # 12 raw -> 9: 1 duplicate, 1 null quantity, 1 return removed
canonical_skus = enriched['canonical_sku'].nunique()  # POPTART-STRAW, WATER-24, POPTART-BLUE (FLASH-AA's only row had no quantity)
idempotent = first.equals(second)                     # Build 5: identical frames from two fresh runs
blueberry_call = ('different flavor is a different product; the case finding was strawberry specifically, '
                  'so merging flavors would erase the signal. PT-12 / SKU#459812 / POPTART-STRAW are one '
                  'product under three codes, which is a different situation.')

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 9
canonical SKUs: 3
same result twice: True
blueberry: different flavor is a different product; the case finding was strawberry specifically, so merging flavors would erase the signal. PT-12 / SKU#459812 / POPTART-STRAW are one product under three codes, which is a different situation.
